## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [200]:
# import libraries
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import joblib
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import mlflow
from mlflow.models import infer_signature
import os

In [201]:
# Load the dataset
df_validation = pd.read_csv("//Users/juancarlosvilar/Documents/GitHub/mlops-and-system-design/session3/dataset/Churn_Modelling_val.csv")

In [202]:
df_validation = df_validation.dropna()

def transform(df: pd.DataFrame) -> pd.DataFrame:


    df.loc[df['HasCrCard'] == 1.0, 'HasCrCard'] = 1
    df.loc[df["HasCrCard"] == 0.0, "HasCrCard"] = 0
    df["HasCrCard"] = df["HasCrCard"].astype(int)

    df.loc[df['IsActiveMember'] == 1.0, 'IsActiveMember'] = 1
    df.loc[df["IsActiveMember"] == 0.0, "IsActiveMember"] = 0
    df["IsActiveMember"] = df["IsActiveMember"].astype(int)

    df.loc[df['Exited'] == 1.0, 'Exited'] = 1
    df.loc[df["Exited"] == 0.0, "Exited"] = 0
    df["Exited"] = df["Exited"].astype(int)

    df.loc[df['Gender'] == 'Female', 'Gender'] = 1
    df.loc[df["Gender"] == 'Male', "Gender"] = 0
    df["Gender"] = df["Gender"].astype(int)

    df["Geography"] = df["Geography"].astype(str)

    df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname',])

    return df




In [203]:
import joblib

PATH = "./"
encoder = OneHotEncoder(drop=None, sparse_output=False).set_output(transform="pandas")
encoder.fit(pd.DataFrame({'Geography': ['France', 'Germany', 'Spain']}))
geography_encoded = encoder.transform(df_validation[['Geography']])
joblib.dump(encoder, f'{PATH}one_hot_encoder.pkl')


['./one_hot_encoder.pkl']

In [204]:
# Perform another experiment if you don't have the ones from session 2. Otherwise, this part can be skipped

# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

##### Batch Inference

In [205]:
# define a function to implement batch inference with mlflow
def batch_inference(model_uri: str, input_df: pd.DataFrame):
    # Load the saved model from MLflow
    model = mlflow.pyfunc.load_model(model_uri)

    # Load the saved encoder
    encoder = joblib.load(f"{PATH}one_hot_encoder.pkl")

    # Copy the input and apply the same preprocessing as training
    df_batch = transform(input_df.copy()).reset_index(drop=True)

    # Encode Geography using the saved encoder
    geography_encoded = encoder.transform(df_batch[['Geography']])

    X_batch = pd.concat([df_batch.drop(columns=['Geography', 'Exited']), geography_encoded], axis=1)

    return model.predict(X_batch)


In [206]:
# define the model uri that should be used
model_uri = 'runs:/d792f1f1d37849f4bffef86e82147e2c/random_forest_model'

batch_prediction_result = batch_inference(model_uri, df_validation)


In [207]:
# check the confusion matrix
from sklearn.metrics import confusion_matrix
y_true = df_validation['Exited']
y_pred = batch_prediction_result
cm = confusion_matrix(y_true, y_pred)
print(cm)

[[623 179]
 [ 51 147]]


##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [208]:
import requests
import json

In [209]:
# import validation dataset to test inference - just one record
df_validation = pd.read_csv('/Users/juancarlosvilar/Documents/GitHub/mlops-and-system-design/session3/dataset/Churn_Modelling_val.csv').head(1)


In [210]:
df_validation

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,8607,15694581,Rawlings,807,Spain,Male,42.0,5,0.0,2,1.0,1.0,74900.9,0


Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [211]:
def get_inference_endpoint(input,host="http://127.0.0.1", port=5000):
        model = mlflow.pyfunc.load_model(model_uri)
        input = transform(input)
        
        return f"{host}:{port}/invocations"

url = get_inference_endpoint(df_validation)


In [212]:
def online_inference_pandas(url: str, input_df: pd.DataFrame):
    encoder = joblib.load(f"{PATH}one_hot_encoder.pkl")

    df = transform(input_df.copy()).reset_index(drop=True)
    
    geography_encoded = encoder.transform(df[['Geography']])
    df = pd.concat([df.drop(columns=['Geography', 'Exited']), geography_encoded], axis=1)

    payload = {
        "dataframe_split": {
            "columns": df.columns.tolist(),
            "data": df.values.tolist(),
        }
    }
    response = requests.post(url, headers={"Content-Type": "application/json"}, json=payload, timeout=30)
    response.raise_for_status()
    return response.json()

response_pandas = online_inference_pandas(url, df_validation)
response_pandas

{'predictions': [0]}

In [217]:
def online_inference_json(url: str, input_df: pd.DataFrame):
    encoder = joblib.load(f"{PATH}one_hot_encoder.pkl")
    
    df = transform(input_df.copy()).reset_index(drop=True)
    geography_encoded = encoder.transform(df[['Geography']])
    df = pd.concat([df.drop(columns=['Geography', 'Exited']), geography_encoded], axis=1)

    payload = {
        "inputs": df.to_dict(orient="list") 
    }

    response = requests.post(url, headers={"Content-Type": "application/json"}, json=payload, timeout=30)
    response.raise_for_status()
    return response.json()

response_json = online_inference_json(url, df_validation)
print(response_json)

{'predictions': [0]}


In [218]:
df_validation

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,8607,15694581,Rawlings,807,Spain,0,42.0,5,0.0,2,1,1,74900.9,0


In [219]:
# define the json as required by MLFlow
input_json = {
    "dataframe_split": {
        "columns": [
            "CreditScore", "Gender", "Age", "Tenure", "Balance",
            "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
            "Geography_France", "Geography_Germany", "Geography_Spain"
            ],
        "data": [
            [807, 0, 42.0, 5, 0.0,2, 1, 1, 74900.90,  0.0, 0.0, 1.0] 
            ]
    }
}

In [220]:
response_json = requests.post(url, headers={"Content-Type": "application/json"}, json=input_json, timeout=30)
print(response_json.json())

{'predictions': [0]}
